# Big Data (PySpark) – Ventas de restaurante
Dataset sintético con el esquema del DW (BD_PUNTOVENTA). Compara pandas vs Spark a 10K, 100K y 1M filas.
Evidencias: EXP-02/EXP-03, E-06, E-08, 6.3.

In [ ]:
!pip -q install pyspark

In [ ]:
import time, statistics, numpy as np, pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[*]").appName("BI_Restaurante")
         .config("spark.sql.shuffle.partitions", "8").getOrCreate())
print("Spark", spark.version)

rng = np.random.default_rng(42)  # semilla fija = reproducible

# Dimensiones (mismas que el DW de prueba)
productos = pd.DataFrame({"idProducto":[1,2,3,4,5],
    "tResumido":["Hamburguesa","Pizza","Inca Kola","Agua","Ceviche"],
    "precio":[18.0,22.0,6.0,4.0,28.0]})
mozos = pd.DataFrame({"idMozo":[1,2,3], "tMozo":["M001","M002","M003"]})
cajas = pd.DataFrame({"idCaja":[1,2], "tCaja":["C01","C02"]})

def gen_ventas(n):
    idp  = rng.integers(1, 6, n)
    cant = rng.integers(1, 4, n)
    precio = productos.set_index("idProducto").loc[idp, "precio"].to_numpy()
    return pd.DataFrame({
        "idVenta": np.arange(1, n+1), "idProducto": idp,
        "idMozo": rng.integers(1, 4, n), "idCaja": rng.integers(1, 3, n),
        "idFecha": rng.integers(1, 366, n), "cantidad": cant,
        "venta": np.round(precio*cant, 2)})

In [ ]:
# Misma consulta analítica en pandas y en Spark: ventas y unidades por producto, mozo y caja
P, M, Cj = (spark.createDataFrame(x) for x in (productos, mozos, cajas))

def q_pandas(path):
    v = pd.read_parquet(path)
    r = v.merge(productos, on="idProducto").merge(mozos, on="idMozo").merge(cajas, on="idCaja")
    return r.groupby(["tResumido","tMozo","tCaja"], as_index=False).agg(
        ventas=("venta","sum"), unidades=("cantidad","sum"))

def q_spark(path):
    v = spark.read.parquet(path)
    r = v.join(P,"idProducto").join(M,"idMozo").join(Cj,"idCaja")
    return (r.groupBy("tResumido","tMozo","tCaja")
             .agg(F.sum("venta").alias("ventas"), F.sum("cantidad").alias("unidades"))
             .toPandas())

def medir(f, path, reps=3):
    t = []
    for _ in range(reps):
        s = time.perf_counter(); out = f(path); t.append(time.perf_counter()-s)
    return statistics.median(t), statistics.pstdev(t), out

# Calentamiento de la JVM (no se reporta)
gen_ventas(1000).to_parquet("/content/warm.parquet"); q_spark("/content/warm.parquet")

In [ ]:
filas = []
for n in [10_000, 100_000, 1_000_000]:
    path = f"/content/ventas_{n}.parquet"
    gen_ventas(n).to_parquet(path)                      # ingesta: dataset procesado en Parquet

    tp, sp, op = medir(q_pandas, path)
    ts, ss, os_ = medir(q_spark, path)

    n_spark = spark.read.parquet(path).count()          # RF-11: validar conteo
    ok_cnt  = (n_spark == n)
    ok_sum  = abs(op["ventas"].sum() - os_["ventas"].sum()) < 0.01   # RF-12: reconciliación
    filas.append({"filas": n, "pandas_s": round(tp,3), "pandas_sd": round(sp,3),
                  "spark_s": round(ts,3), "spark_sd": round(ss,3),
                  "conteo_ok": ok_cnt, "suma_ok": ok_sum,
                  "ventas_total": round(os_["ventas"].sum(),2)})
res = pd.DataFrame(filas)
res

In [ ]:
# Resultado agregado en Parquet (RF-10) y gráfico para el informe (6.4)
q = (spark.read.parquet("/content/ventas_1000000.parquet")
     .join(P,"idProducto").join(M,"idMozo").join(Cj,"idCaja")
     .groupBy("tResumido","tMozo","tCaja")
     .agg(F.sum("venta").alias("ventas"), F.sum("cantidad").alias("unidades")))
q.write.mode("overwrite").parquet("/content/resultado_agregado_parquet")

import matplotlib.pyplot as plt
ax = res.plot(x="filas", y=["pandas_s","spark_s"], marker="o", logx=True)
ax.set_xlabel("Filas (escala log)"); ax.set_ylabel("Tiempo mediano (s)")
ax.set_title("Consulta ventas por producto/mozo/caja: pandas vs Spark (3 repeticiones)")
plt.grid(True); plt.show()
res.to_csv("/content/resultados_experimento.csv", index=False)